# Step 1: Load Data into SQLite & Run Analytical Queries

**What we're doing:** Taking the raw Telco Churn CSV, cleaning known data
quality issues, loading it into a local SQLite database, and running 20
analytical SQL queries against it.

**Why:** This is the project's SQL chapter. SQLite is a real relational
database (just file-based, no server needed) — every query here uses
syntax that works identically against Postgres, MySQL, or Snowflake.

**Colab setup note:** Colab's filesystem resets every session. Either
(a) re-upload the CSV each session, or (b) mount Google Drive so your
`churn.db` file and notebooks persist between sessions. Drive is recommended.

In [1]:
# Mount Google Drive so your work persists across Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

# Set this to wherever you want the project folder to live in your Drive.
PROJECT_ROOT = '/content/drive/MyDrive/churn-platform'

import os
os.makedirs(f'{PROJECT_ROOT}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/processed', exist_ok=True)
print('Upload WA_Fn-UseC_-Telco-Customer-Churn.csv into:', f'{PROJECT_ROOT}/data/raw')

Mounted at /content/drive
Upload WA_Fn-UseC_-Telco-Customer-Churn.csv into: /content/drive/MyDrive/churn-platform/data/raw


In [2]:
import pandas as pd
import sqlite3
from pathlib import Path

RAW_CSV_PATH = f'{PROJECT_ROOT}/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
DB_PATH = f'{PROJECT_ROOT}/data/processed/churn.db'

def load_and_clean(csv_path):
    df = pd.read_csv(csv_path)

    # TotalCharges arrives as text with 11 blank strings (new customers,
    # tenure=0, haven't been billed yet). Coerce to numeric; blanks -> 0.
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)

    # Convert target from 'Yes'/'No' strings to clean 0/1 integers.
    df['Churn'] = (df['Churn'] == 'Yes').astype(int)

    return df

df = load_and_clean(RAW_CSV_PATH)
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [3]:
# Quick sanity check before we trust this data in SQL queries.
print('Nulls per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('Churn rate:', df['Churn'].mean().round(4))
print('TotalCharges dtype:', df['TotalCharges'].dtype)

Nulls per column:
Series([], dtype: int64)

Churn rate: 0.2654
TotalCharges dtype: float64


In [4]:
# Write to SQLite. if_exists='replace' makes this notebook safely re-runnable.
conn = sqlite3.connect(DB_PATH)
df.to_sql('customers', conn, if_exists='replace', index=False)
conn.execute('CREATE INDEX IF NOT EXISTS idx_churn ON customers(Churn);')
conn.commit()
print(f'Loaded {len(df)} rows into {DB_PATH}')

Loaded 7043 rows into /content/drive/MyDrive/churn-platform/data/processed/churn.db


## Now run the analytical queries

We wrote 20 queries across 4 files in `sql/`. Below, we run a representative
sample directly in the notebook so the results are visible. Run the rest by
copying queries from the `.sql` files into `pd.read_sql(query, conn)`.

In [5]:
# Q1: Overall churn rate
query = '''
SELECT
    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate_pct
FROM customers;
'''
pd.read_sql(query, conn)

,total_customers,churned_customers,churn_rate_pct
0,7043,1869,26.54


In [6]:
# Q2: Churn rate by contract type
query = '''
SELECT
    Contract,
    COUNT(*) AS num_customers,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC;
'''
pd.read_sql(query, conn)

,Contract,num_customers,churn_rate_pct
0,Month-to-month,3875,42.71
1,One year,1473,11.27
2,Two year,1695,2.83


In [7]:
# Q7: Tenure cohort analysis
query = '''
SELECT
    CASE
        WHEN tenure <= 6 THEN '0-6 months'
        WHEN tenure <= 12 THEN '7-12 months'
        WHEN tenure <= 24 THEN '13-24 months'
        WHEN tenure <= 48 THEN '25-48 months'
        ELSE '49+ months'
    END AS tenure_cohort,
    COUNT(*) AS num_customers,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY tenure_cohort
ORDER BY MIN(tenure);
'''
cohort_df = pd.read_sql(query, conn)
cohort_df

,tenure_cohort,num_customers,churn_rate_pct
0,0-6 months,1481,52.94
1,7-12 months,705,35.89
2,13-24 months,1024,28.71
3,25-48 months,1594,20.39
4,49+ months,2239,9.51


In [8]:
# Q14: Window function + CTE — tenure quartile churn rate
query = '''
WITH tenure_quartiles AS (
    SELECT
        customerID, tenure, Churn,
        NTILE(4) OVER (ORDER BY tenure) AS tenure_quartile
    FROM customers
)
SELECT
    tenure_quartile,
    COUNT(*) AS num_customers,
    ROUND(100.0 * SUM(Churn) / COUNT(*), 2) AS churn_rate_pct
FROM tenure_quartiles
GROUP BY tenure_quartile
ORDER BY tenure_quartile;
'''
pd.read_sql(query, conn)

,tenure_quartile,num_customers,churn_rate_pct
0,1,1761,50.37
1,2,1761,29.02
2,3,1761,18.91
3,4,1760,7.84


## Save a clean processed CSV for the next steps

EDA, stats, and modeling notebooks will load from this file rather than
re-cleaning the raw CSV each time — keeps every later step consistent.

In [9]:
df.to_csv(f'{PROJECT_ROOT}/data/processed/churn_clean.csv', index=False)
print('Saved cleaned dataset for downstream steps.')
conn.close()

Saved cleaned dataset for downstream steps.
